# Reconnaissance d'expressions faciales — FER2013

**Projet Deep Learning** — Étudiant A / Étudiant B

**Objectif :** entraîner et comprendre un modèle qui reconnaît 7 expressions faciales
(angry, disgust, fear, happy, neutral, sad, surprise) à partir d'images de visages 48×48 en niveaux de gris.

**Dataset :** FER2013 — Kaggle `msambare/fer2013`

**Plan du notebook** (construit progressivement, une cellule par micro-étape) :
1. Exploration et préparation du dataset
2. Baseline dense
3. CNN
4. Entraînement
5. Évaluation et analyse des erreurs
6. Expériences (≥ 3)


Avant de charger les données, on inspecte l'arborescence de `data/fer2013` : chaque dossier de classe sert d'étiquette (label), c'est cette structure que Keras lira.

In [ ]:
# Micro-étape 1a : explorer la structure du dataset FER2013
from pathlib import Path

DATA_DIR = Path("data/fer2013")  # chemin relatif à la racine du projet

for split in ["train", "test"]:  # les deux splits du dataset
    total = 0
    # on trie les classes par nom pour un affichage stable et lisible
    for classe in sorted(p for p in (DATA_DIR / split).iterdir() if p.is_dir()):
        n = len(list(classe.glob("*.jpg")))  # nombre d'images .jpg dans ce dossier de classe
        total += n
        print(f"{split}/{classe.name} : {n}")
    print(f"{split} total : {total}\n")  # total du split

### Étape 1b — Regarder les images et vérifier le format

**Pourquoi ?** Un modèle apprend à partir de pixels : s'il reçoit des images de tailles ou de modes différents, la couche `Flatten` produira des vecteurs de longueurs différentes et l'entraînement échouera (ou buggera silencieusement).

On fait donc deux choses :
1. **On affiche une grille de 7 classes × 5 images** tirées de `data/fer2013/train/<classe>/` : on vérifie *à l'œil* que les dossiers contiennent bien des visages et que les classes sont cohérentes (utile aussi pour l'oral : montrer des exemples réels).
2. **On vérifie le format sur *toutes* les images** (train + test) avec PIL : taille `(48, 48)`, mode `L` (niveaux de gris) et extension `.jpg`. On liste les exceptions plutôt que de les faire échouer en silence.

> Librairies utilisées uniquement : `PIL`, `matplotlib` et `os` — aucune dépendance supplémentaire.


In [ ]:
# Micro-étape 1b : exemples d'images + vérification du format (taille, mode, extension)
import os
import matplotlib.pyplot as plt
from PIL import Image

DATA_DIR = os.path.join("data", "fer2013")  # racine du dataset (chemin relatif à la racine du projet)
N_EXEMPLES = 5  # nombre d'images affichées par classe
# les 7 classes dans l'ordre canonique : cet ordre sera repris pour l'encodage one-hot (étape 1c)
CLASSES = ["angry", "disgust", "fear", "happy", "neutral", "sad", "surprise"]

# --- 1) grille 7 lignes (classes) x 5 colonnes (exemples) -------------------
fig, axes = plt.subplots(len(CLASSES), N_EXEMPLES, figsize=(10, 14))
for ligne, classe in enumerate(CLASSES):
    dossier = os.path.join(DATA_DIR, "train", classe)
    # on trie les noms de fichiers pour que la grille soit reproductible à chaque exécution
    fichiers = sorted(f for f in os.listdir(dossier) if f.endswith(".jpg"))[:N_EXEMPLES]
    for col, nom in enumerate(fichiers):
        with Image.open(os.path.join(dossier, nom)) as img:  # le `with` ferme l'image automatiquement
            axes[ligne, col].imshow(img, cmap="gray")  # cmap="gray" car les images sont en niveaux de gris
        axes[ligne, col].set_xticks([])  # on masque les graduations : on veut voir l'image, pas les axes
        axes[ligne, col].set_yticks([])
    # nom de la classe écrit à gauche de la ligne (une seule fois)
    axes[ligne, 0].set_ylabel(classe, rotation=0, ha="right", va="center")

fig.suptitle("FER2013 — 5 exemples par classe (split train)", fontsize=14)
plt.tight_layout()
plt.show()

# --- 2) vérification du format sur TOUTES les images (train + test) --------
attendu = ((48, 48), "L", ".jpg")  # taille, mode PIL (L = niveaux de gris), extension
stats_taille, stats_mode, stats_ext = {}, {}, {}
anomalies = []
n_images = 0

for split in ["train", "test"]:
    racine_split = os.path.join(DATA_DIR, split)
    for classe in sorted(os.listdir(racine_split)):
        dossier = os.path.join(racine_split, classe)
        if not os.path.isdir(dossier):  # on ignore les fichiers éventuels à la racine du split
            continue
        for nom in sorted(os.listdir(dossier)):
            chemin = os.path.join(dossier, nom)
            n_images += 1
            ext = os.path.splitext(nom)[1].lower()
            with Image.open(chemin) as img:
                taille, mode = img.size, img.mode
            # on compte les valeurs observées pour pouvoir afficher un bilan honnête
            stats_taille[taille] = stats_taille.get(taille, 0) + 1
            stats_mode[mode] = stats_mode.get(mode, 0) + 1
            stats_ext[ext] = stats_ext.get(ext, 0) + 1
            if (taille, mode, ext) != attendu:
                anomalies.append((chemin, taille, mode, ext))

print(f"Images vérifiées : {n_images} (train + test)")
print(f"Tailles observées    : {stats_taille}")
print(f"Modes PIL observés   : {stats_mode}")
print(f"Extensions observées : {stats_ext}")
if anomalies:
    # critère de validation non rempli : on liste les exceptions au lieu de les cacher
    print(f"\n{len(anomalies)} exception(s) trouvée(s) sur {n_images} images :")
    for chemin, taille, mode, ext in anomalies[:20]:  # on n'affiche que les 20 premières
        print(f"  - {chemin} -> taille {taille}, mode {mode}, extension {ext}")
    if len(anomalies) > 20:
        print(f"  ... et {len(anomalies) - 20} autre(s)")
else:
    print(f"\nFormat : 48×48, mode L, .jpg — vérifié sur {n_images} images, aucune exception.")
